# Comparison Country Ranks Pilot Indicator and Revised Indicator

This notebook compares the results of the pilot and of the final vulnerability indicator at the country level. It loads the country-level mean vulnerability scores from the pilot indicator version and from the updated indicator and aligns the datasets using country ISO codes. The notebook standardizes the data structure, handles missing or invalid values, and ensures that vulnerability scores are comparable across both datasets.

The workflow then calculates country rankings based on mean vulnerability scores for each indicator and evaluates the differences between the two versions. It quantifies both the absolute change in vulnerability scores and the change in country rankings. In addition, the notebook computes correlation metrics to assess the overall differences between the indicators. It creates:

- csv showing the differences in vulnerability scores and ranking positions between the two indicators

- csv with summary statistics describing the magnitude of changes across countries with correlation measures (Pearson and Spearman) to evaluate the differences between the two indicator versions

## How to run
1. Put the required input files in the same folder as this notebook (or edit the paths in the **Configuration** cell below).
2. Run the cells from top to bottom.

## Required files
- `country_vulnerability_stats.csv`
- `country_combined_means.csv`

In [ ]:
#import packages
import numpy as np
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from adjustText import adjust_text  


In [ ]:
# paths
indicator_1_path = Path('C:\\Users\\janab\\Documents\\Jupyter\\vulnerability_indicator\\countries\\country_vulnerability_stats.csv')
indicator_0_path = Path("country_combined_means.csv")                 

#load data
old = pd.read_csv(indicator_0_path, usecols=["ISO", "COUNTRY", "mean_vulnerability"]).copy()
new = pd.read_csv(indicator_1_path, usecols=["country_id", "iso", "name", "mean_vulnerability"]).copy()

# --- standardize keys / column names ---
old = old.rename(columns={"ISO": "iso", "COUNTRY": "name", "mean_vulnerability": "mean_vulnerability_old"})
new = new.rename(columns={"mean_vulnerability": "mean_vulnerability_new"})

old["iso"] = old["iso"].astype(str).str.strip().str.upper()
new["iso"] = new["iso"].astype(str).str.strip().str.upper()

# negative mean_vulnerability: treat as 0 (negative vulnerability not possible (comes from old concept: E+S-A=V
old["mean_vulnerability_old"] = pd.to_numeric(old["mean_vulnerability_old"], errors="coerce").clip(lower=0)
new["mean_vulnerability_new"] = pd.to_numeric(new["mean_vulnerability_new"], errors="coerce").clip(lower=0)
#add rank columns
old["rank_old"] = old["mean_vulnerability_old"].rank(ascending=False, method="min")
new["rank_new"] = new["mean_vulnerability_new"].rank(ascending=False, method="min")

# merge and calculate differences
df = old.merge(
    new[["iso", "country_id", "name", "mean_vulnerability_new", "rank_new"]],
    on="iso",
    how="left",
    suffixes=("", "_newname")
)
#calculate vulnerarbility difference
df["delta_v"] = df["mean_vulnerability_new"] - df["mean_vulnerability_old"]
df["abs_delta_v"] = df["delta_v"].abs()
#calculate rank difference
df["delta_rank"] = df["rank_new"] - df["rank_old"]
df["abs_delta_rank"] = df["delta_rank"].abs()

# correlations 
valid_scores = df[["mean_vulnerability_old", "mean_vulnerability_new"]].dropna()
valid_ranks  = df[["rank_old", "rank_new"]].dropna()

pearson_r = float(valid_scores.corr(method="pearson").iloc[0, 1]) if len(valid_scores) >= 2 else np.nan
spearman_rho = float(valid_ranks.corr(method="spearman").iloc[0, 1]) if len(valid_ranks) >= 2 else np.nan

summary = {
    "old_file": indicator_0_path.name,
    "new_file": indicator_1_path.name,
    "mean_abs_delta_v": float(df["abs_delta_v"].mean(skipna=True)),
    "max_abs_delta_v": float(df["abs_delta_v"].max(skipna=True)),
    "mean_abs_delta_rank": float(df["abs_delta_rank"].mean(skipna=True)),
    "max_abs_delta_rank": float(df["abs_delta_rank"].max(skipna=True)),
    "pearson_r": pearson_r,
    "spearman_rho": spearman_rho,
    "n_compared_scores": int(len(valid_scores)),
    "n_compared_ranks": int(len(valid_ranks)),
    "n_old": int(old["iso"].nunique()),
    "n_new": int(new["iso"].nunique()),
    "n_matched": int(df["mean_vulnerability_new"].notna().sum()),
}

# save outputs
out_path = "diff_mean_vulnerability_indicator_1_vs_indicator_0.csv"
out_summary_csv = "summary_mean_vulnerability_indicator_1_vs_indicator_0.csv"

df.to_csv(out_path, index=False)
pd.DataFrame([summary]).to_csv(out_summary_csv, index=False)
